# 03 — Seed Dataset Generation

## Frozen pipeline position

This notebook creates the individual-level **seed dataset** for Notebook 04.

**NFHS-5 prevalence → DA-WI relationships/weights → individual scenarios → logical constraints → app-specific safety variables → risk labels → seed dataset**

### Methodological boundary
- NFHS-5 here is an aggregated state/UT reference dataset, so it anchors prevalence rather than acting as respondent-level training data.
- DA-WI provides the published 26-factor risk structure and weights.
- App-specific variables such as `safe_now`, `perpetrator_present`, `can_leave_safely`, `medical_help`, and `contact_requested` are not claimed to come from NFHS or DA-WI.
- Seed labels are synthetic scenario labels for prototype development, not clinical or real-world ground truth.
- CTGAN is **not** used in this notebook.


In [3]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)

NFHS_PATH = '../data/processed/nfhs_violence_reference.csv'
DAWI_PATH = '../data/processed/dawi_risk_factors.csv'
MAP_PATH = '../data/processed/dawi_nfhs_mapping.csv'

nfhs = pd.read_csv(NFHS_PATH)
dawi = pd.read_csv(DAWI_PATH)
mapping = pd.read_csv(MAP_PATH)

print('NFHS:', nfhs.shape)
print('DA-WI:', dawi.shape)
print('Mapping:', mapping.shape)


NFHS: (111, 4)
DA-WI: (26, 7)
Mapping: (26, 9)


## 1. Clean NFHS prevalence values

The source contains values such as `(0.0)`. We convert these to numeric percentages without modifying the original reference files.


In [5]:
# ============================================================
# CLEAN NFHS PREVALENCE VALUES
# ============================================================

raw_values = (
    nfhs['NFHS-5 (2019-20)']
    .astype(str)
    .str.strip()
)

# Remove formatting characters such as (0.0)
clean_values = (
    raw_values
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

# Convert non-numeric values such as *, NA, - to NaN
nfhs['value_pct'] = pd.to_numeric(
    clean_values,
    errors='coerce'
)

print("Original values:", len(raw_values))
print("Successfully converted:", nfhs['value_pct'].notna().sum())
print("Missing/non-numeric:", nfhs['value_pct'].isna().sum())

print("\nNon-numeric source values found:")
print(raw_values[~raw_values.str.match(r'^-?\d+(\.\d+)?$', na=False)]
      .value_counts())

print("\nCleaned value statistics:")
print(nfhs['value_pct'].describe())

display(nfhs.head())

Original values: 111
Successfully converted: 110
Missing/non-numeric: 1

Non-numeric source values found:
NFHS-5 (2019-20)
(0.0)     2
(11.7)    1
(6.1)     1
(13.1)    1
(0.4)     1
(3.2)     1
*         1
Name: count, dtype: int64

Cleaned value statistics:
count    110.000000
mean       8.088182
std       10.581356
min        0.000000
25%        1.025000
50%        2.650000
75%       11.200000
max       44.500000
Name: value_pct, dtype: float64


,Indicators,sub indicators,NFHS-5 (2019-20),STATE/UT,value_pct
0,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,23.2,Andaman & Nicobar Islands,23.2
1,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,(0.0),Andaman & Nicobar Islands,0.0
2,Gender Based Violence (age 18-49 years),Young women age 18-29 years who experienced se...,1.4,Andaman & Nicobar Islands,1.4
3,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ev...,28.8,Andhra Pradesh,28.8
4,Gender Based Violence (age 18-49 years),Ever-married women age 18-49 years who have ex...,3.5,Andhra Pradesh,3.5


## 2. Build state-level NFHS prevalence anchors

The downloaded table provides three relevant violence indicators:

- ever experienced spousal violence
- physical violence during any pregnancy
- sexual violence by age among young women

These are **population prevalence anchors**, not individual labels.


In [6]:
def get_nfhs_indicator(pattern):
    mask = nfhs['sub indicators'].astype(str).str.contains(
        pattern, case=False, na=False
    )
    out = nfhs.loc[mask, ['STATE/UT', 'value_pct']].copy()
    return out.rename(columns={'value_pct': 'prevalence_pct'})

spousal_prev = get_nfhs_indicator('spousal violence')
pregnancy_prev = get_nfhs_indicator('physical violence during any pregnancy')
sexual_prev = get_nfhs_indicator('sexual violence')

print('Spousal:', spousal_prev.shape)
print('Pregnancy:', pregnancy_prev.shape)
print('Sexual:', sexual_prev.shape)


Spousal: (37, 2)
Pregnancy: (37, 2)
Sexual: (37, 2)


In [7]:
state_prev = (
    spousal_prev.rename(columns={'prevalence_pct': 'spousal_violence_pct'})
    .merge(
        pregnancy_prev.rename(columns={'prevalence_pct': 'pregnancy_violence_pct'}),
        on='STATE/UT', how='outer'
    )
    .merge(
        sexual_prev.rename(columns={'prevalence_pct': 'sexual_violence_pct'}),
        on='STATE/UT', how='outer'
    )
)

print('States/UTs in reference:', len(state_prev))
display(state_prev.head(10))


States/UTs in reference: 37


,STATE/UT,spousal_violence_pct,pregnancy_violence_pct,sexual_violence_pct
0,Andaman & Nicobar Islands,23.2,0.0,1.4
1,Andhra Pradesh,28.8,3.5,3.8
2,Arunachal Pradesh,18.5,1.1,0.1
3,Assam,26.6,2.2,7.4
4,Bihar,40.6,1.9,7.1
5,Chandigarh,9.7,0.0,NaN
6,Chhattisgarh,14.0,1.2,0.5
7,Dadra & Nagar Haveli and Daman & Diu,21.8,1.3,4.4
8,Goa,6.0,0.9,1.6
9,Gujarat,10.0,2.2,3.0


## 3. Generate the individual-level seed population

We create 10,000 synthetic individual scenarios. State is sampled uniformly across the available state/UT reference rows; NFHS prevalence then anchors the relevant violence variables.

For DA-WI factors not present in the NFHS CSV, the generator uses transparent scenario-generation probabilities that are correlated with a latent severity variable. These are **not claimed epidemiological estimates**.


In [8]:
N_SEED = 10000

states = state_prev['STATE/UT'].dropna().unique()
state_probs = np.ones(len(states)) / len(states)

seed = pd.DataFrame({
    'STATE/UT': np.random.choice(states, size=N_SEED, p=state_probs)
})

seed = seed.merge(state_prev, on='STATE/UT', how='left')

seed['age'] = np.clip(
    np.round(np.random.normal(31, 8, N_SEED)),
    18, 49
).astype(int)

print(seed.shape)
display(seed.head())


(10000, 5)


,STATE/UT,spousal_violence_pct,pregnancy_violence_pct,sexual_violence_pct,age
0,Jammu & Kashmir,5.9,0.3,1.4,19
1,Uttarakhand,12.5,2.4,0.0,22
2,Puducherry,29.8,1.3,0.0,34
3,Meghalaya,23.2,2.5,7.9,22
4,Chandigarh,9.7,0.0,NaN,40


## 4. Generate correlated DA-WI risk factors

We avoid independent random bits. A latent severity signal controls related factors. The published DA-WI weights are used later for the reference score; they are **not converted into fake prevalence estimates**.


In [9]:
severity = np.random.beta(2.0, 5.0, N_SEED)

spousal_context = np.clip(
    seed['spousal_violence_pct'].fillna(24.2) / 100.0,
    0.01, 0.80
)

preg_context = np.clip(
    seed['pregnancy_violence_pct'].fillna(2.5) / 100.0,
    0.001, 0.20
)

def bernoulli(p):
    return (
        np.random.random(len(seed)) <
        np.clip(p, 0.001, 0.99)
    ).astype(int)

seed['violence_escalation'] = bernoulli(
    0.10 + 0.35 * severity + 0.35 * spousal_context
)

seed['threat_to_kill'] = bernoulli(
    0.01 + 0.08 * severity + 0.08 * seed['violence_escalation']
)
seed['violent_jealousy'] = bernoulli(0.04 + 0.18 * severity)
seed['recent_separation'] = bernoulli(0.05 + 0.10 * severity)
seed['lethal_weapon'] = bernoulli(0.01 + 0.05 * severity)
seed['avoids_arrest'] = bernoulli(0.01 + 0.05 * severity)
seed['strangulation'] = bernoulli(
    0.005 + 0.045 * severity + 0.02 * seed['violence_escalation']
)
seed['illegal_drug_use'] = bernoulli(0.02 + 0.08 * severity)
seed['problem_drinking'] = bernoulli(0.02 + 0.10 * severity)

seed['violence_during_pregnancy'] = bernoulli(
    np.clip(preg_context + 0.02 * severity, 0.001, 0.20)
)

seed['partner_capable_of_killing'] = bernoulli(0.03 + 0.12 * severity)
seed['suicide_threat_attempt'] = bernoulli(0.01 + 0.05 * severity)
seed['withholds_necessities'] = bernoulli(0.03 + 0.15 * severity)
seed['intimidating_behavior'] = bernoulli(0.06 + 0.22 * severity)
seed['rumors'] = bernoulli(0.03 + 0.10 * severity)
seed['false_accusations'] = bernoulli(0.03 + 0.12 * severity)
seed['family_rejection'] = bernoulli(0.02 + 0.08 * severity)
seed['social_isolation'] = bernoulli(0.04 + 0.18 * severity)
seed['inlaws_support_abuse'] = bernoulli(0.02 + 0.08 * severity)
seed['infertility_related_abuse'] = bernoulli(0.005 + 0.04 * severity)
seed['healthcare_neglect'] = bernoulli(0.01 + 0.08 * severity)
seed['family_honor_threat'] = bernoulli(0.01 + 0.05 * severity)
seed['leaving_threat'] = bernoulli(
    0.02 + 0.12 * severity + 0.05 * seed['recent_separation']
)
seed['hide_abuse'] = bernoulli(0.04 + 0.16 * severity)
seed['lack_of_support'] = bernoulli(0.05 + 0.15 * severity)
seed['family_supports_abuse'] = bernoulli(0.02 + 0.08 * severity)

print('Generated 26 DA-WI factors.')


Generated 26 DA-WI factors.


## 5. Add app-specific immediate-safety variables

These represent the minimal information available to the on-device safety assessment. They are separate from DA-WI and NFHS.


In [10]:
seed['perpetrator_present'] = bernoulli(
    0.10 + 0.35 * severity + 0.10 * seed['violence_escalation']
)

seed['safe_now'] = bernoulli(
    0.90 - 0.55 * severity - 0.20 * seed['perpetrator_present']
)

seed['can_leave_safely'] = bernoulli(
    0.88
    - 0.40 * severity
    - 0.25 * seed['perpetrator_present']
    - 0.15 * seed['social_isolation']
)

seed['medical_help'] = bernoulli(
    0.02
    + 0.12 * severity
    + 0.25 * seed['violence_during_pregnancy']
    + 0.15 * seed['strangulation']
)

seed['contact_requested'] = bernoulli(
    0.10
    + 0.25 * severity
    + 0.25 * (1 - seed['safe_now'])
)

seed['immediate_threat'] = (
    (seed['safe_now'] == 0) &
    (seed['perpetrator_present'] == 1)
).astype(int)

display(
    seed[
        [
            'safe_now',
            'perpetrator_present',
            'can_leave_safely',
            'medical_help',
            'contact_requested',
            'immediate_threat'
        ]
    ].head()
)


,safe_now,perpetrator_present,can_leave_safely,medical_help,contact_requested,immediate_threat
0,0,0,1,0,0,0
1,1,0,1,0,1,0
2,1,0,1,0,0,0
3,1,0,1,0,0,0
4,1,0,1,0,1,0


## 6. Calculate the DA-WI reference score

The published weighted DA-WI has a theoretical score range of 0–64. This score is retained for analysis and validation; it is not the final app model output.


In [11]:
DAWI_WEIGHTS = dict(
    zip(dawi['feature'], dawi['weight'])
)

risk_columns = list(DAWI_WEIGHTS.keys())

seed['dawi_reference_score'] = sum(
    seed[col] * weight
    for col, weight in DAWI_WEIGHTS.items()
)

print(seed['dawi_reference_score'].describe())
print('Maximum possible score:', sum(DAWI_WEIGHTS.values()))


count    10000.00000
mean         3.61630
std          3.05431
min          0.00000
25%          2.00000
50%          3.00000
75%          5.00000
max         20.00000
Name: dawi_reference_score, dtype: float64
Maximum possible score: 64


## 7. Create prototype risk labels

Two targets are retained:

- `severe_ipv_risk`: synthetic evidence-informed reference outcome.
- `immediate_safety_risk`: operational app target based on current immediate danger.

The immediate-safety target intentionally lets current danger override historical/reference severity.


In [12]:
score_norm = seed['dawi_reference_score'] / 64.0

severe_prob = np.clip(
    0.02
    + 0.65 * score_norm
    + 0.12 * seed['threat_to_kill']
    + 0.15 * seed['strangulation']
    + 0.10 * seed['lethal_weapon'],
    0.001,
    0.98
)

seed['severe_ipv_risk'] = (
    np.random.random(N_SEED) < severe_prob
).astype(int)

high_condition = (
    (
        (seed['safe_now'] == 0) &
        (seed['perpetrator_present'] == 1) &
        (seed['can_leave_safely'] == 0)
    )
    |
    (
        (seed['medical_help'] == 1) &
        (seed['immediate_threat'] == 1)
    )
    |
    (
        (seed['lethal_weapon'] == 1) &
        (seed['immediate_threat'] == 1)
    )
)

medium_condition = (
    (seed['safe_now'] == 0)
    |
    (seed['perpetrator_present'] == 1)
    |
    (seed['can_leave_safely'] == 0)
    |
    (seed['medical_help'] == 1)
)

seed['immediate_safety_risk'] = np.select(
    [high_condition, medium_condition],
    ['HIGH', 'MEDIUM'],
    default='LOW'
)

print('Immediate safety distribution:')
display(seed['immediate_safety_risk'].value_counts())

print('Severe IPV reference-label proportion:')
print(seed['severe_ipv_risk'].value_counts(normalize=True))


Immediate safety distribution:


immediate_safety_risk
MEDIUM    5181
LOW       4222
HIGH       597
Name: count, dtype: int64

Severe IPV reference-label proportion:
severe_ipv_risk
0    0.9324
1    0.0676
Name: proportion, dtype: float64


## 8. Quality and logical consistency checks

In [13]:
binary_columns = risk_columns + [
    'safe_now',
    'perpetrator_present',
    'can_leave_safely',
    'medical_help',
    'contact_requested',
    'immediate_threat',
    'severe_ipv_risk'
]

missing = seed[binary_columns].isna().sum().sum()

binary_valid = all(
    set(seed[c].unique()).issubset({0, 1})
    for c in binary_columns
)

invalid_immediate = (
    (seed['immediate_threat'] == 1)
    &
    ~(
        (seed['safe_now'] == 0)
        &
        (seed['perpetrator_present'] == 1)
    )
).sum()

print('Missing model values:', missing)
print('All binary variables valid:', binary_valid)
print('Invalid immediate-threat records:', invalid_immediate)


Missing model values: 0
All binary variables valid: True
Invalid immediate-threat records: 0


In [14]:
print('Immediate safety distribution:')
display(
    seed['immediate_safety_risk']
    .value_counts(normalize=True)
    .rename('proportion')
)

print('DA-WI reference score by immediate risk:')
display(
    seed.groupby('immediate_safety_risk')['dawi_reference_score']
    .agg(['count', 'mean', 'median', 'max'])
)

print('NFHS spousal-violence prevalence across generated states:')
display(
    seed.groupby('STATE/UT')['spousal_violence_pct']
    .first()
    .describe()
)


Immediate safety distribution:


immediate_safety_risk
MEDIUM    0.5181
LOW       0.4222
HIGH      0.0597
Name: proportion, dtype: float64

DA-WI reference score by immediate risk:


,count,mean,median,max
immediate_safety_risk,,,,
HIGH,597,4.569514,4.0,19
LOW,4222,3.207958,2.0,16
MEDIUM,5181,3.839220,4.0,20


NFHS spousal-violence prevalence across generated states:


count    37.000000
mean     19.972973
std      10.534432
min       1.000000
25%      11.300000
50%      21.800000
75%      26.600000
max      44.500000
Name: spousal_violence_pct, dtype: float64

## 9. Save seed dataset

In [15]:
os.makedirs('../data/processed', exist_ok=True)

seed_path = '../data/processed/seed_risk_dataset.csv'

seed.to_csv(seed_path, index=False)

print('Saved:', seed_path)
print('Shape:', seed.shape)

display(seed.head())


Saved: ../data/processed/seed_risk_dataset.csv
Shape: (10000, 40)


,STATE/UT,spousal_violence_pct,pregnancy_violence_pct,sexual_violence_pct,age,violence_escalation,threat_to_kill,violent_jealousy,recent_separation,lethal_weapon,...,family_supports_abuse,perpetrator_present,safe_now,can_leave_safely,medical_help,contact_requested,immediate_threat,dawi_reference_score,severe_ipv_risk,immediate_safety_risk
0,Jammu & Kashmir,5.9,0.3,1.4,19,0,0,0,0,0,...,0,0,0,1,0,0,0,6,0,MEDIUM
1,Uttarakhand,12.5,2.4,0.0,22,0,0,0,1,0,...,0,0,1,1,0,1,0,6,0,LOW
2,Puducherry,29.8,1.3,0.0,34,1,0,0,0,0,...,0,0,1,1,0,0,0,4,0,LOW
3,Meghalaya,23.2,2.5,7.9,22,1,0,0,0,0,...,0,0,1,1,0,0,0,2,0,LOW
4,Chandigarh,9.7,0.0,NaN,40,0,0,0,0,0,...,0,0,1,1,0,1,0,2,0,LOW


## Frozen output for Notebook 04

`../data/processed/seed_risk_dataset.csv`

Notebook 04 will use this seed dataset for CTGAN generation, synthetic-data quality checks, logical-consistency checks, and final dataset export.

**No model training is performed in Notebook 03.**


In [16]:
# ============================================================
# NOTEBOOK 03 — FINAL VALIDATION OUTPUT
# Run this cell and send me the complete output
# ============================================================

print("=" * 80)
print("1. DATASET")
print("=" * 80)

print("Seed dataset shape:", seed.shape)
print("Saved file exists:", os.path.exists(
    "../data/processed/seed_risk_dataset.csv"
))

print("\nColumns:")
print(seed.columns.tolist())


print("\n" + "=" * 80)
print("2. IMMEDIATE SAFETY RISK DISTRIBUTION")
print("=" * 80)

print(
    seed["immediate_safety_risk"]
    .value_counts()
    .sort_index()
)

print("\nProportions:")
print(
    seed["immediate_safety_risk"]
    .value_counts(normalize=True)
    .sort_index()
)


print("\n" + "=" * 80)
print("3. SEVERE IPV REFERENCE LABEL")
print("=" * 80)

print(seed["severe_ipv_risk"].value_counts())
print("\nProportion:")
print(seed["severe_ipv_risk"].mean())


print("\n" + "=" * 80)
print("4. DA-WI SCORE")
print("=" * 80)

print(seed["dawi_reference_score"].describe())

print("\nScore by immediate risk:")
display(
    seed.groupby("immediate_safety_risk")[
        "dawi_reference_score"
    ].agg(["count", "mean", "median", "min", "max"])
)


print("\n" + "=" * 80)
print("5. LOGICAL CONSISTENCY")
print("=" * 80)

binary_columns = risk_columns + [
    "safe_now",
    "perpetrator_present",
    "can_leave_safely",
    "medical_help",
    "contact_requested",
    "immediate_threat",
    "severe_ipv_risk"
]

print(
    "Missing binary values:",
    seed[binary_columns].isna().sum().sum()
)

print(
    "Invalid binary values:",
    sum(
        not set(seed[col].unique()).issubset({0, 1})
        for col in binary_columns
    )
)

invalid_immediate = (
    (seed["immediate_threat"] == 1)
    &
    ~(
        (seed["safe_now"] == 0)
        &
        (seed["perpetrator_present"] == 1)
    )
).sum()

print(
    "Invalid immediate-threat records:",
    invalid_immediate
)


print("\n" + "=" * 80)
print("6. SAMPLE")
print("=" * 80)

display(
    seed[
        [
            "STATE/UT",
            "age",
            "dawi_reference_score",
            "safe_now",
            "perpetrator_present",
            "can_leave_safely",
            "medical_help",
            "contact_requested",
            "immediate_threat",
            "severe_ipv_risk",
            "immediate_safety_risk"
        ]
    ].head(10)
)

1. DATASET
Seed dataset shape: (10000, 40)
Saved file exists: True

Columns:
['STATE/UT', 'spousal_violence_pct', 'pregnancy_violence_pct', 'sexual_violence_pct', 'age', 'violence_escalation', 'threat_to_kill', 'violent_jealousy', 'recent_separation', 'lethal_weapon', 'avoids_arrest', 'strangulation', 'illegal_drug_use', 'problem_drinking', 'violence_during_pregnancy', 'partner_capable_of_killing', 'suicide_threat_attempt', 'withholds_necessities', 'intimidating_behavior', 'rumors', 'false_accusations', 'family_rejection', 'social_isolation', 'inlaws_support_abuse', 'infertility_related_abuse', 'healthcare_neglect', 'family_honor_threat', 'leaving_threat', 'hide_abuse', 'lack_of_support', 'family_supports_abuse', 'perpetrator_present', 'safe_now', 'can_leave_safely', 'medical_help', 'contact_requested', 'immediate_threat', 'dawi_reference_score', 'severe_ipv_risk', 'immediate_safety_risk']

2. IMMEDIATE SAFETY RISK DISTRIBUTION
immediate_safety_risk
HIGH       597
LOW       4222
MEDIUM

,count,mean,median,min,max
immediate_safety_risk,,,,,
HIGH,597,4.569514,4.0,0,19
LOW,4222,3.207958,2.0,0,16
MEDIUM,5181,3.839220,4.0,0,20



5. LOGICAL CONSISTENCY
Missing binary values: 0
Invalid binary values: 0
Invalid immediate-threat records: 0

6. SAMPLE


,STATE/UT,age,dawi_reference_score,safe_now,perpetrator_present,can_leave_safely,medical_help,contact_requested,immediate_threat,severe_ipv_risk,immediate_safety_risk
0,Jammu & Kashmir,19,6,0,0,1,0,0,0,0,MEDIUM
1,Uttarakhand,22,6,1,0,1,0,1,0,0,LOW
2,Puducherry,34,4,1,0,1,0,0,0,0,LOW
3,Meghalaya,22,2,1,0,1,0,0,0,0,LOW
4,Chandigarh,40,2,1,0,1,0,1,0,0,LOW
5,Chandigarh,30,0,1,0,1,0,0,0,0,LOW
6,Arunachal Pradesh,32,6,1,0,1,0,1,0,0,LOW
7,Telangana,29,9,0,0,0,0,1,0,0,MEDIUM
8,Meghalaya,37,0,1,0,1,0,1,0,0,LOW
9,Odisha,37,2,0,1,0,0,1,1,0,HIGH
